# ⚡ Tutorial 3: Supervised Fine-Tuning (SFT) & Direct Preference Optimization (DPO) Alignment

Welcome to the **Kilat Post-Training & Alignment** tutorial! In this notebook, you will learn how to:
1. **Prepare a Kilat model for post-training** (`prepare_kilat_for_post_training`): Deactivate KV caching (`use_cache=False`), attach input-embedding gradient hooks (`enable_input_require_grads()`), and enable **non-reentrant activation checkpointing** (`use_reentrant=False`).
2. **Mask user prompt tokens with `-100`** (`KilatCompletionOnlyCollator`): Uphold **Golden Invariant #3 (Causal & Data Leakage Integrity)** so cross-entropy loss is computed strictly on assistant completions and never on user instructions or padding.
3. **Run Supervised Fine-Tuning (SFT)** with PEFT LoRA adapters.
4. **Align the model with Direct Preference Optimization (DPO)**: Optimize chosen vs. rejected completions using `compute_sequence_logprobs` and `compute_dpo_preference_loss` (Rafailov et al., 2023) and verify that the implicit reward margin $r_\theta(x, y_w) - r_\theta(x, y_l)$ becomes strictly positive.

In [ ]:
# Uncomment to install in Google Colab:
# !pip install -q git+https://github.com/Airukua/Kilat.git peft trl transformers safetensors

## 1. Prepare `KilatForCausalLM` + PEFT LoRA for Post-Training

`prepare_kilat_for_post_training()` configures:
- `model.config.use_cache = False`
- `enable_input_require_grads()` (essential when `wte` is frozen by LoRA and gradient checkpointing is active)
- `gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})`

In [ ]:
import torch
from peft import get_peft_model

from kilat.integrations.huggingface import (
    KilatCompletionOnlyCollator,
    compute_dpo_preference_loss,
    compute_sequence_logprobs,
    get_kilat_lora_config,
    prepare_kilat_for_post_training,
    register_kilat,
)
from kilat.integrations.huggingface.configuration_kilat import KilatConfig
from kilat.integrations.huggingface.modeling_kilat import KilatForCausalLM

torch.manual_seed(42)
register_kilat()

cfg = KilatConfig(
    vocab_size=128,
    n_embd=64,
    n_layer=2,
    n_head=4,
    recall_ratio=0.5,
    recall_pattern="macro_synergy",
    ffn_mode="swiglu",
    use_rope=True,
    dropout=0.0,
)

base_model = KilatForCausalLM(cfg)
lora_cfg = get_kilat_lora_config(r=8, lora_alpha=16, lora_dropout=0.0)
model = get_peft_model(base_model, lora_cfg)
prepare_kilat_for_post_training(model, use_gradient_checkpointing=True, use_reentrant=False)

print("use_cache:", model.config.use_cache)
print("is_gradient_checkpointing:", base_model.is_gradient_checkpointing)

## 2. Completion-Only Prompt Loss Masking (`-100`) & SFT Training

`KilatCompletionOnlyCollator` locates `response_template_ids` (e.g., the tokenized representation of `<|assistant|>`) and masks every token up to and including the template—as well as right-padded positions—with `-100`.

In [ ]:
RESPONSE_TEMPLATE_IDS = [10, 11]  # Example token IDs for '<|assistant|>'

collator = KilatCompletionOnlyCollator(
    response_template_ids=RESPONSE_TEMPLATE_IDS,
    pad_token_id=0,
    max_length=16,
)

sft_samples = [
    {"input_ids": [1, 21, 22, 10, 11, 41, 42, 43, 2]},
    {"input_ids": [1, 25, 10, 11, 51, 52, 2]},
]
sft_batch = collator(sft_samples)

print("Input IDs:\n", sft_batch["input_ids"])
print("Masked Labels (-100 on prompt & padding):\n", sft_batch["labels"])

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-2)
model.train()
for step in range(10):
    optimizer.zero_grad(set_to_none=True)
    out = model(input_ids=sft_batch["input_ids"], labels=sft_batch["labels"])
    out.loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step == 0 or (step + 1) % 5 == 0:
        print(f"SFT Step {step + 1:02d} | Completion CE Loss: {out.loss.item():.4f}")

## 3. Direct Preference Optimization (DPO) Alignment

In DPO (Rafailov et al., 2023), we align the policy $\pi_\theta$ against a frozen reference policy $\pi_{\text{ref}}$ on preference pairs $(x, y_w, y_l)$ without training a separate reward model:

$$\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l)}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)\right]$$

In [ ]:
chosen_batch = collator([
    {"input_ids": [1, 21, 22, 10, 11, 41, 42, 43, 2]},
    {"input_ids": [1, 25, 26, 10, 11, 51, 52, 53, 2]},
])
rejected_batch = collator([
    {"input_ids": [1, 21, 22, 10, 11, 81, 82, 83, 2]},
    {"input_ids": [1, 25, 26, 10, 11, 91, 92, 93, 2]},
])

# Snapshot reference log-probabilities before DPO updates
with torch.no_grad():
    ref_chosen_logps = compute_sequence_logprobs(model, chosen_batch["input_ids"], chosen_batch["labels"])
    ref_rejected_logps = compute_sequence_logprobs(model, rejected_batch["input_ids"], rejected_batch["labels"])

dpo_optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-2)
for step in range(8):
    dpo_optimizer.zero_grad(set_to_none=True)
    pi_chosen = compute_sequence_logprobs(model, chosen_batch["input_ids"], chosen_batch["labels"])
    pi_rejected = compute_sequence_logprobs(model, rejected_batch["input_ids"], rejected_batch["labels"])
    dpo_loss, chosen_rew, rejected_rew = compute_dpo_preference_loss(
        pi_chosen,
        pi_rejected,
        ref_chosen_logps,
        ref_rejected_logps,
        beta=0.2,
    )
    dpo_loss.backward()
    dpo_optimizer.step()
    margin = float((chosen_rew - rejected_rew).mean().item())
    if step == 0 or (step + 1) % 4 == 0:
        print(f"DPO Step {step + 1:02d} | Loss: {dpo_loss.item():.4f} | Reward Margin (chosen - rejected): {margin:+.4f}")